# IMPORTING THE REQUIRED PACKAGES

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mplsoccer import Pitch, Sbopen, VerticalPitch
import mplsoccer
import sympy as sp
from scipy.interpolate import griddata
import numpy as np
import matplotlib.animation as animation
from IPython.display import HTML
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import io
from PIL import Image
import requests
from bs4 import BeautifulSoup
from statsbombpy import sb
import json
import csv
import networkx as nx

# LOADING THE DATA

In [2]:
#IMPORTING THE FULL THING AND SEEING WHICH DATASETS ARE AVAILABLE
comps=sb.competitions()
comps['competition_name'].unique()

array(['1. Bundesliga', 'African Cup of Nations', 'Champions League',
       'Copa America', 'Copa del Rey', "FA Women's Super League",
       'FIFA U20 World Cup', 'FIFA World Cup', 'Frauen Bundesliga',
       'Indian Super league', 'La Liga', 'Liga F', 'Liga Profesional',
       'Ligue 1', 'Major League Soccer', 'North American League', 'NWSL',
       'Premier League', 'Serie A', 'Serie A Women', 'UEFA Euro',
       'UEFA Europa League', "UEFA Women's Euro", "Women's World Cup"],
      dtype=object)

In [6]:
#SELECTING THE COMPETETION ACCORDING TO THE PROBLEM
comps_pro=comps[comps['competition_name']=='FIFA World Cup']
comps_pro

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
30,43,106,International,FIFA World Cup,male,False,True,2022,2026-05-04T01:48:57.914346,2026-05-04T01:53:40.309717,2026-05-04T01:53:40.309717,2026-05-04T01:48:57.914346
31,43,3,International,FIFA World Cup,male,False,True,2018,2026-05-15T15:48:12.789440,2021-06-13T16:17:31.694,None,2026-05-15T15:48:12.789440
32,43,55,International,FIFA World Cup,male,False,True,1990,2023-06-28T10:58:20.137929,2021-06-12T16:17:31.694,None,2023-06-28T10:58:20.137929
33,43,54,International,FIFA World Cup,male,False,True,1986,2026-05-04T01:30:23.212111,2021-06-13T16:17:31.694,None,2026-05-04T01:30:23.212111
34,43,51,International,FIFA World Cup,male,False,True,1974,2026-05-15T02:20:43.439689,2021-06-13T16:17:31.694,None,2026-05-15T02:20:43.439689
35,43,272,International,FIFA World Cup,male,False,True,1970,2026-05-04T02:09:10.916859,None,None,2026-05-04T02:09:10.916859
36,43,270,International,FIFA World Cup,male,False,True,1962,2026-05-04T02:05:35.653887,None,None,2026-05-04T02:05:35.653887
37,43,269,International,FIFA World Cup,male,False,True,1958,2026-05-04T02:02:41.853159,None,None,2026-05-04T02:02:41.853159


In [7]:
#INSPECTING MATCH ID AND SEASON ID TO SELECT THE MATCH/MATCHES YOU NEED
matches=sb.matches(competition_id=43,season_id=106)
matches

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,...,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3857276,2022-12-01,15:00:00.000,International - FIFA World Cup,2022,Canada,Morocco,1,2,available,...,2026-05-04T01:50:33.318123,3,Group Stage,Al Thumama Stadium,Raphael Claus,John Herdman,Walid Regragui,1.1.0,2,2
1,3857271,2022-11-21,13:00:00.000,International - FIFA World Cup,2022,England,Iran,6,2,available,...,2026-05-04T01:43:30.216315,1,Group Stage,Sheikh Khalifa International Stadium,Raphael Claus,Gareth Southgate,Carlos Manuel Brito Leal Queiróz,1.1.0,2,2
2,3857296,2022-12-01,15:00:00.000,International - FIFA World Cup,2022,Croatia,Belgium,0,0,available,...,2026-05-04T01:50:06.343225,3,Group Stage,Ahmad bin Ali Stadium,Anthony Taylor,Zlatko Dalić,Roberto Martínez Montoliú,1.1.0,2,2
3,3857274,2022-11-25,16:00:00.000,International - FIFA World Cup,2022,Netherlands,Ecuador,1,1,available,...,2026-05-04T01:50:07.646458,2,Group Stage,Sheikh Khalifa International Stadium,Mustapha Ghorbal,Louis van Gaal,Gustavo Julio Alfaro,1.1.0,2,2
4,3857255,2022-12-01,19:00:00.000,International - FIFA World Cup,2022,Japan,Spain,2,1,available,...,2026-05-04T01:49:36.594944,3,Group Stage,Sheikh Khalifa International Stadium,Victor Miguel de Freitas Gomes,Hajime Moriyasu,Luis Enrique Martínez García,1.1.0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,3857291,2022-11-23,16:00:00.000,International - FIFA World Cup,2022,Spain,Costa Rica,7,0,available,...,2026-05-03T16:12:13.412295,1,Group Stage,Al Thumama Stadium,NaN,Luis Enrique Martínez García,Luis Fernando Suárez Guzman,1.1.0,2,2
60,3857281,2022-11-27,16:00:00.000,International - FIFA World Cup,2022,Croatia,Canada,4,1,available,...,2026-05-04T01:51:52.104873,2,Group Stage,Sheikh Khalifa International Stadium,Andrés Matonte,Zlatko Dalić,John Herdman,1.1.0,2,2
61,3857280,2022-12-02,19:00:00.000,International - FIFA World Cup,2022,Cameroon,Brazil,1,0,available,...,2026-05-03T15:44:07.018244,3,Group Stage,Lusail Stadium,Ismail Elfath,Rigobert Song Bahanag,Adenor Leonardo Bacchi,1.1.0,2,2
62,3857279,2022-11-22,19:00:00.000,International - FIFA World Cup,2022,France,Australia,4,1,available,...,2026-05-04T01:50:33.615237,1,Group Stage,Al Janoub Stadium,Victor Miguel de Freitas Gomes,Didier Deschamps,Graham James Arnold,1.1.0,2,2


In [8]:
#ARGENTINA MATCH DIRECTORY 2022
#AUSTRALIA-3869151
#FRANCE-3869685
#MEXICO-3857289
#CROATIA-3869519
#SAUDI-3857300
#NETHERLANDS-3869321
#POLAND-3857264

In [9]:
#SELECTING THE TEAMS ENTIRE MATCHES DURING THE TOURNAMENT
matches_AGH=matches[(matches['home_team']=='Argentina')]
matches_AGA=matches[(matches['away_team']=='Argentina')]
AG_all=[matches_AGH,matches_AGA]
matches_AG=pd.concat(AG_all)
matches_AG

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,...,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
27,3869685,2022-12-18,17:00:00.000,International - FIFA World Cup,2022,Argentina,France,3,3,available,...,2024-12-16T10:21:13.710934,7,Final,Lusail Stadium,Szymon Marciniak,Lionel Sebastián Scaloni,Didier Deschamps,1.1.0,2,2
30,3869151,2022-12-03,21:00:00.000,International - FIFA World Cup,2022,Argentina,Australia,2,1,available,...,2023-10-15T17:09:05.163199,4,Round of 16,Ahmad bin Ali Stadium,Szymon Marciniak,Lionel Sebastián Scaloni,Graham James Arnold,1.1.0,2,2
39,3857289,2022-11-26,19:00:00.000,International - FIFA World Cup,2022,Argentina,Mexico,2,0,available,...,2026-05-04T01:49:14.941205,2,Group Stage,Lusail Stadium,Daniele Orsato,Lionel Sebastián Scaloni,Gerardo Daniel Martino,1.1.0,2,2
42,3869519,2022-12-13,21:00:00.000,International - FIFA World Cup,2022,Argentina,Croatia,3,0,available,...,2023-04-26T22:32:37.808359,6,Semi-finals,Lusail Stadium,Daniele Orsato,Lionel Sebastián Scaloni,Zlatko Dalić,1.1.0,2,2
56,3857300,2022-11-22,10:00:00.000,International - FIFA World Cup,2022,Argentina,Saudi Arabia,1,2,available,...,2026-05-04T01:51:00.290088,1,Group Stage,Lusail Stadium,Slavko Vinčić,Lionel Sebastián Scaloni,Hervé Renard,1.1.0,2,2
24,3869321,2022-12-09,19:00:00.000,International - FIFA World Cup,2022,Netherlands,Argentina,2,2,available,...,2025-09-07T08:42:38.594465,5,Quarter-finals,Lusail Stadium,Antonio Miguel Mateu Lahoz,Louis van Gaal,Lionel Sebastián Scaloni,1.1.0,2,2
38,3857264,2022-11-30,19:00:00.000,International - FIFA World Cup,2022,Poland,Argentina,0,2,available,...,2026-05-03T16:16:14.789834,3,Group Stage,Stadium 974,Danny Desmond Makkelie,Czesław Michniewicz,Lionel Sebastián Scaloni,1.1.0,2,2


In [83]:
#SELECTING THE MATCH
parser = Sbopen()
df, related, freeze, tactics = parser.event(3869685	)
#get team names
team1, team2 = df.team_name.unique()



In [84]:
#SELECTING A TEAM
df=df[df['team_name']=='France']

# SELECTING AND BIFURCATING DIFFERENT ACTIONS 

In [85]:
shots = df[df['type_name']=='Shot']
passes = df[df['type_name']=='Pass'] 
ball_Rec=df[df['type_name']=='Ball Recovery']
ball_receive=df[df['type_name']=='Ball Receipt']
duel=df[df['type_name']=='Duel']
carry=df[df['type_name']=='Carry']
dribble=df[df['type_name']=='Dribble']
pressure=df[df['type_name']=='Pressure']


In [86]:
shotsARG=shots[shots['team_name']=='Argentina']


In [87]:
df['type_name'].unique()

array(['Starting XI', 'Half Start', 'Pass', 'Ball Receipt', 'Carry',
       'Pressure', 'Foul Committed', 'Ball Recovery', 'Foul Won',
       'Miscontrol', 'Clearance', 'Interception', 'Duel', 'Goal Keeper',
       'Dribble', 'Dribbled Past', 'Dispossessed', 'Block',
       'Injury Stoppage', 'Substitution', 'Tactical Shift', 'Shield',
       '50/50', 'Half End', 'Shot', 'Bad Behaviour'], dtype=object)

# 1. SPATIAL ENTROPY OF ACTIONS

## FUNCTION TO CALUCULATE ENTROPY

In [88]:
# 1. Define Pep's 20-zone grid boundaries (StatsBomb 120x80 coordinate system)
X_BINS = [0, 30, 60, 102, 120]       # 4 horizontal bands
Y_BINS = [0, 18, 30, 50, 62, 80]     # 5 vertical lanes

# 2. Calculate the exact area of each of the 20 zones
dx = np.diff(X_BINS)  # [30, 30, 42, 18]
dy = np.diff(Y_BINS)  # [18, 12, 20, 12, 18]

# Create a 4x5 matrix of zone areas, flattened to 20 zones
zone_areas = np.outer(dx, dy).flatten()
total_pitch_area = zone_areas.sum()  # 9600 sq units



In [89]:
def compute_action_spatial_entropy(df, x_col='x', y_col='y'):
    """
    Assigns events to Pep's 20 zones and computes spatial entropy metrics.
    """
    # Drop missing coordinates and clip to pitch bounds
    coords = df[[x_col, y_col]].dropna().copy()
    x = np.clip(coords[x_col].values, 0, 119.999)
    y = np.clip(coords[y_col].values, 0, 79.999)
    
    # Digitize into 0-indexed bins (0..3 for x, 0..4 for y)
    x_idx = np.digitize(x, X_BINS) - 1
    y_idx = np.digitize(y, Y_BINS) - 1
    
    # Map 2D grid coordinates to a single zone ID (0 to 19)
    zone_ids = x_idx * (len(Y_BINS) - 1) + y_idx
    
    # Count actions per zone (ensuring all 20 zones are represented)
    counts = np.bincount(zone_ids, minlength=20)
    total_actions = counts.sum()
    
    if total_actions == 0:
        return {"n_events": 0, "shannon_norm": np.nan, "batty_entropy": np.nan, "batty_norm": np.nan}
    
    # Empirical probability p_i for each zone
    p = counts / total_actions
    
    # Filter to non-zero zones to avoid log(0)
    mask = p > 0
    p_valid = p[mask]
    areas_valid = zone_areas[mask]
    
    # A. Standard Shannon Entropy (normalized to [0, 1])
    h_shannon = -np.sum(p_valid * np.log(p_valid))
    h_shannon_norm = h_shannon / np.log(20)
    
    # B. Batty's Spatial Entropy: -sum(p_i * ln(p_i / A_i))
    # Max possible Batty entropy occurs when p_i = A_i / Total_Area, yielding ln(Total_Area)
    h_batty = -np.sum(p_valid * np.log(p_valid / areas_valid))
    h_batty_norm = h_batty / np.log(total_pitch_area)
    
    return {
        "n_events": int(total_actions),
        "shannon_norm": round(h_shannon_norm, 4),
        "batty_entropy": round(h_batty, 4),
        "batty_norm": round(h_batty_norm, 4)
    }



In [90]:
# 3. Run across all your action DataFrames
action_dfs = {
    "Shots": shots,
    "Passes": passes,
    "Ball Recoveries": ball_Rec,
    "Ball Receives": ball_receive,
    "Duels": duel,
    "Carries": carry,
    "Dribbles": dribble,
    "Pressures": pressure
}

results = []
for action_name, df in action_dfs.items():
    metrics = compute_action_spatial_entropy(df, x_col='x', y_col='y')
    metrics["Action"] = action_name
    results.append(metrics)

summary_df = pd.DataFrame(results)[["Action", "n_events", "shannon_norm", "batty_entropy", "batty_norm"]]
print(summary_df)

            Action  n_events  shannon_norm  batty_entropy  batty_norm
0            Shots        14        0.3433         6.9867      0.7620
1           Passes       570        0.8944         8.9811      0.9795
2  Ball Recoveries        56        0.8719         8.9229      0.9731
3    Ball Receives       489        0.8842         8.9457      0.9756
4            Duels        48        0.8896         8.9267      0.9735
5          Carries       427        0.8679         8.8986      0.9705
6         Dribbles        33        0.7299         8.3411      0.9097
7        Pressures       194        0.9126         9.0480      0.9867
